# LinguaForge — GGUF Q4_K_M export

Merges the LinguaForge LoRA into the base Gemma 4 E4B weights, converts the
resulting model to GGUF, quantises it to **Q4_K_M**, and benchmarks CPU
decode tokens / sec inside the Kaggle T4 instance.

Outputs (all under `/kaggle/working/`):
- `merged_fp16/` — merged FP16 transformers checkpoint
- `gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf` — quantised GGUF
- `gguf/Modelfile` — ready-to-run Ollama Modelfile
- `gguf/cpu_bench.json` — CPU decode benchmark

In [ ]:
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
assert os.environ.get('HF_TOKEN'), 'HF_TOKEN missing — set as env var or Kaggle Secret before running.'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('HF token loaded:', os.environ['HF_TOKEN'][:8], '...', os.environ['HF_TOKEN'][-4:])

In [ ]:
!pip install -qU unsloth==2026.5.2 peft transformers accelerate sentencepiece hf_transfer
!pip install -q gguf

## 1. Merge LoRA into base Gemma 4 E4B (FP16)

Use Unsloth's `save_pretrained_merged` so the merged weights match the
tokenizer layout the LoRA was trained against.

In [ ]:
import glob, os
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Mounted Kaggle dataset = same adapter, no HF Hub round-trip required.
candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
assert candidates, 'Adapter dataset not mounted — add dongwei666/linguaforge-gemma4-204lang-lora as a data source.'
ADAPTER_DIR = os.path.dirname(candidates[0])
print('ADAPTER_DIR =', ADAPTER_DIR)
# /kaggle/working is only ~20 GB; merged FP16 + GGUF f16 is ~32 GB.
# Route the big intermediates through /tmp (~73 GB) and keep only the
# final Q4_K_M + Modelfile + bench under /kaggle/working.
MERGED_DIR = '/tmp/merged_fp16'
GGUF_TMP_DIR = '/tmp/gguf'
os.makedirs(GGUF_TMP_DIR, exist_ok=True)
os.makedirs('/kaggle/working/gguf', exist_ok=True)

model, tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=True,
    token=os.environ['HF_TOKEN'],
)
tok = get_chat_template(tok, chat_template='gemma')
print('model + adapter loaded')

In [ ]:
# Merge LoRA + write FP16 checkpoint suitable for llama.cpp conversion.
model.save_pretrained_merged(MERGED_DIR, tok, save_method='merged_16bit')
!ls -lah {MERGED_DIR} | head -n 20

## 2. Build llama.cpp

In [ ]:
!cd /kaggle/working && git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!apt-get install -qq -y cmake build-essential libcurl4-openssl-dev > /tmp/apt.log 2>&1 || true
!cd /kaggle/working/llama.cpp && cmake -B build -DLLAMA_CURL=OFF -DGGML_NATIVE=ON > /tmp/cmake.log 2>&1
!cd /kaggle/working/llama.cpp && cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli > /tmp/build.log 2>&1
!ls /kaggle/working/llama.cpp/build/bin | head

## 3. Convert merged FP16 → GGUF (FP16)


In [ ]:
!cd /kaggle/working/llama.cpp && python convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf --outtype f16 2>&1 | tail -n 30

## 4. Quantise to Q4_K_M

In [ ]:
!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    {GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf \
    /kaggle/working/gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf Q4_K_M
# Free up the f16 intermediate immediately so disk doesn't fill mid-pipeline.
import os
if os.path.exists(f'{GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf'):
    os.remove(f'{GGUF_TMP_DIR}/linguaforge-gemma4-e4b.f16.gguf')
!ls -lah /kaggle/working/gguf

## 5. CPU benchmark inside Kaggle

In [ ]:
import json, re, subprocess, time
GGUF_PATH = '/kaggle/working/gguf/linguaforge-gemma4-e4b.Q4_K_M.gguf'
PROMPT = """<start_of_turn>user\nTranslate this English sentence into Cherokee (Iroquoian, North America):\n\nThe river remembers every footstep on its bank.\n<end_of_turn>\n<start_of_turn>model\n"""

t0 = time.time()
result = subprocess.run([
    '/kaggle/working/llama.cpp/build/bin/llama-cli',
    '-m', GGUF_PATH,
    '-no-cnv',  # raw prompt mode
    '-n', '96',
    '-t', '4',
    '-p', PROMPT,
    '--seed', '1',
], capture_output=True, text=True, timeout=600)
elapsed = time.time() - t0
print('--- stdout tail ---')
print(result.stdout[-800:])
print('--- stderr tail ---')
print(result.stderr[-800:])

# Parse decode throughput from stderr ("llama_print_timings" or "eval time" lines).
match = re.search(r'eval time =\s*([0-9.]+) ms /\s*(\d+) (?:tokens|runs)\s*\(\s*([0-9.]+) ms per token,\s*([0-9.]+) tokens per second', result.stderr)
metrics = {'elapsed_s': elapsed, 'tokens_per_sec_cpu': None}
if match:
    metrics['tokens_per_sec_cpu'] = float(match.group(4))
    metrics['eval_tokens'] = int(match.group(2))
with open('/kaggle/working/gguf/cpu_bench.json', 'w') as f:
    json.dump(metrics, f, indent=2)
metrics

## 6. Ollama Modelfile

In [ ]:
MODELFILE = '''\
FROM ./linguaforge-gemma4-e4b.Q4_K_M.gguf

TEMPLATE \"\"\"<start_of_turn>user
{{ if .System }}{{ .System }}
{{ end }}{{ .Prompt }}<end_of_turn>
<start_of_turn>model
{{ .Response }}<end_of_turn>
\"\"\"

PARAMETER stop \"<start_of_turn>\"
PARAMETER stop \"<end_of_turn>\"
PARAMETER temperature 0.7
PARAMETER top_p 0.9

SYSTEM \"\"\"You are LinguaForge, a multilingual tutor for endangered and low-resource languages.
Stay accurate, concise, and faithful to the target script. Always include
an English gloss in parentheses next to any native phrase.
\"\"\"
'''
with open('/kaggle/working/gguf/Modelfile', 'w') as f:
    f.write(MODELFILE)
print(open('/kaggle/working/gguf/Modelfile').read())

In [ ]:
# Final disk hygiene: keep only the Q4_K_M GGUF, Modelfile, and bench JSON
# under /kaggle/working so the published kernel output stays small.
import shutil, os
for p in [
    '/kaggle/working/llama.cpp',
    MERGED_DIR,
    GGUF_TMP_DIR,
]:
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True)
    elif os.path.isfile(p):
        os.remove(p)
!ls -lah /kaggle/working /kaggle/working/gguf